In [30]:
import torch
from torch import nn
from torch.nn.modules import loss
from torch.utils import data
from torch.utils.data import DataLoader, dataset
from torchvision import datasets
from torchvision.transforms import v2
from argparse import ArgumentParser

In [31]:
# Define Model (specifically a Feed Forward NN)
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


In [32]:
def train(
    dataloader: DataLoader,
    model: nn.Module,
    loss_fn: nn.CrossEntropyLoss,
    optimizer: torch.optim.SGD,
):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropogation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
        # print(f"loss: {loss:>7f}    [{current:>5d}/{size:>5d}]")
    print(f"loss: {loss:>7f}    [{current:>5d}/{size:>5d}]")

def test(dataloader: DataLoader, model: nn.Module, loss_fn: nn.CrossEntropyLoss):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)

            pred = model(X)
            test_loss += loss_fn(pred, y)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        test_loss /= num_batches
        correct /= size
        # print(
        #    f"Test Error: \n Accuracy: {(100 * correct):>0.1f}%, \
        #    Avg loss: {test_loss:>8f} \n"
        # )

        print(
            f"Test Error: \n Accuracy: {(100 * correct):>0.1f}%, \
            Avg loss: {test_loss:>8f} \n"
        )



In [33]:
################ Data Loading with Pytorch ########################

# Pytorch has two primitives to work with data:
#     - torch.utils.data.DataLoader and torch.utils.data.Dataset
#     Dataset stores the samples and their corresponding labels
#     while DataLoader wraps an iterable around the Dataset

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

batch_size = 64

# create data loaders
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of Y: {y.shape} {y.dtype}")
    break


Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of Y: torch.Size([64]) torch.int64


In [34]:
################ Create a Model ########################

# I am using a ternary if-else statement to check if I have an accelerator
# (ie GPU) otherwise use cpu
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"Using {device} device")

model = NeuralNet().to(device)
print(model)


Using cuda device
NeuralNet(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [35]:
################ Optimizing the Model Params ########################

# Need to pick a loss func and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

epochs = 5
for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

torch.save(model.state_dict(), "model.pth")
print("Saved Pytorch Model State to model.pth")

Epoch 1
-------------------------------
loss: 2.159920    [57664/60000]
Test Error: 
 Accuracy: 43.8%,             Avg loss: 2.156766 

Epoch 2
-------------------------------
loss: 1.908784    [57664/60000]
Test Error: 
 Accuracy: 57.6%,             Avg loss: 1.884102 

Epoch 3
-------------------------------
loss: 1.568982    [57664/60000]
Test Error: 
 Accuracy: 62.2%,             Avg loss: 1.517636 

Epoch 4
-------------------------------
loss: 1.307209    [57664/60000]
Test Error: 
 Accuracy: 63.3%,             Avg loss: 1.252127 

Epoch 5
-------------------------------
loss: 1.149681    [57664/60000]
Test Error: 
 Accuracy: 64.6%,             Avg loss: 1.087768 

Done!
Saved Pytorch Model State to model.pth


In [36]:
################ Load and run the Model ########################

model = NeuralNet().to(device)
print(model)
model.load_state_dict(torch.load("model.pth", weights_only=True))

classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

NeuralNet(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)
Predicted: "Ankle boot", Actual: "Ankle boot"
